# Skyvonyx - Colab GPU worker

Watches a Google Drive job queue and runs the staged pipeline on each job:

```
plan (SAHI slice)  ->  detect (YOLOv8)  ->  stitch (SAHI GreedyNMM)  ->  segment (SAM 2)  ->  export (GeoJSON/COCO)
      CPU                 GPU                     CPU                       GPU                    CPU
                     detector resident        detector EVICTED         SAM 2 resident
```

**One expert in VRAM at a time.** YOLOv8 runs over the whole scene and is unloaded before SAM 2 loads.
That is what lets the full stack fit a free T4. Every stage result is written to disk (`memory/`), so a
session disconnect resumes at the interrupted stage instead of starting over.

```
MyDrive/skyvonyx/jobs/inbox/<job_id>/   input/*.tif + job.json + READY (created last)
         -> processing/ -> review/  (output/: *_wgs84.geojson, *_native.geojson, *_coco.json, preview, REPORT_DRAFT.md, summary.json)
         -> a human creates APPROVED -> done/      (REJECTED -> failed/)
```

**Rules.** The worker never emails or shares anything - a human approves every delivery. No API keys in this notebook.
**Licences.** ultralytics/YOLOv8 AGPL-3.0; DOTA-trained weights academic terms; SAM 2 Apache-2.0; SAHI MIT. Fine for pilots, not yet for a commercial service.

Run cells top to bottom. Cell 5 (offline tests) and Cell 6 (GPU smoke test) must pass before you start the loop.

In [ ]:
#@title 1. Runtime check
import subprocess, sys, torch
print(sys.version.split()[0], "| torch", torch.__version__, "| CUDA", torch.version.cuda)
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"GPU: {torch.cuda.get_device_name(0)}  free {free/1e9:.1f} / {total/1e9:.1f} GB")
else:
    print("NO GPU - Runtime > Change runtime type > T4 GPU. The pipeline runs on CPU but SAM 2 will be very slow.")

In [ ]:
#@title 2. Code + dependencies (about 3-5 min on a fresh runtime)
import os, subprocess, pathlib

# Where the backend comes from. Push this repo to GitHub and set the URL, or leave the
# URL empty and keep a copy of the repo at MyDrive/skyvonyx/code (Cell 3 mounts Drive first in that case).
SKYVONYX_REPO = "https://github.com/harishrajCEO/skyvonyx.git"  #@param {type:"string"}
WORK = pathlib.Path("/content/skyvonyx")

def sh(cmd):
    print("$", cmd); subprocess.run(cmd, shell=True, check=True)

if not WORK.exists():
    if SKYVONYX_REPO:
        sh(f"git clone --depth 1 {SKYVONYX_REPO} {WORK}")
    else:
        raise SystemExit("set SKYVONYX_REPO, or run Cell 3 first and copy MyDrive/skyvonyx/code to /content/skyvonyx")

# Upstream repos the stack is built on (shallow clones; the forks are the ones you maintain).
VENDOR = WORK / "vendor"; VENDOR.mkdir(exist_ok=True)
for repo in ["harishrajCEO/sahi", "harishrajCEO/rf-detr", "facebookresearch/sam2",
             "facebookresearch/segment-anything"]:
    dst = VENDOR / repo.split("/")[1]
    if not dst.exists():
        sh(f"git clone --depth 1 -q https://github.com/{repo}.git {dst}")

# SAM 2's optional CUDA extension often fails to build on Colab and is not needed for
# image prediction; skipping it is the documented fallback.
os.environ["SAM2_BUILD_CUDA"] = "0"
sh(f"pip install -q -r {WORK}/requirements-colab.txt")
sh(f"pip install -q -e {VENDOR}/sahi")          # your SAHI fork
sh(f"pip install -q -e {VENDOR}/sam2")          # SAM 2
sh(f"pip install -q -e {VENDOR}/segment-anything")   # SAM 1 fallback
# RF-DETR is optional (licence escape route). Uncomment when you have a fine-tune to run:
# sh(f"pip install -q -e {VENDOR}/rf-detr")

import sys; sys.path.insert(0, str(WORK))
import skyvonyx, sahi, ultralytics
print("skyvonyx", skyvonyx.__version__, "| sahi", getattr(sahi, "__version__", "?"), "| ultralytics", ultralytics.__version__)

In [ ]:
#@title 3. Mount Drive and set the queue root
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
ROOT = Path("/content/drive/MyDrive/skyvonyx")   #@param {type:"string"}
ROOT = Path(ROOT)
from skyvonyx.queue_worker import ensure_layout, queue_state
ensure_layout(ROOT)
(ROOT / "models").mkdir(exist_ok=True)
print("queue:", queue_state(ROOT))

In [ ]:
#@title 4. Model weights
import urllib.request, pathlib
MODELS = ROOT / "models"

# Detector: your DOTA fine-tune if it exists, else the COCO yolov8n (knows "boat", not "ship").
DETECTOR_WEIGHTS = str(MODELS / "best.pt") if (MODELS / "best.pt").exists() else "yolov8n.pt"
print("detector weights:", DETECTOR_WEIGHTS)
if DETECTOR_WEIGHTS == "yolov8n.pt":
    print("  (no fine-tune at", MODELS / "best.pt", "- using COCO weights. Expect poor satellite results.)")

# SAM 2.1 hiera-small: 184 MB, the right size for a T4. Cached on Drive so it downloads once.
SAM2_CKPT = MODELS / "sam2.1_hiera_small.pt"
if not SAM2_CKPT.exists():
    url = "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_small.pt"
    print("downloading", url); urllib.request.urlretrieve(url, SAM2_CKPT)
print("sam2 checkpoint:", SAM2_CKPT, f"{SAM2_CKPT.stat().st_size/1e6:.0f} MB")

In [ ]:
#@title 5. Offline tests (fake experts, no GPU) - must print ALL ... CHECKS PASSED
!cd /content/skyvonyx && python tests/test_pipeline.py 2>&1 | tail -15

In [ ]:
#@title 6. GPU smoke test: real YOLOv8 + real SAM 2 on a small georeferenced scene
# Proves the GPU path end to end and prints the scheduler report, so you can see the
# detector load, get evicted, and SAM 2 load - with VRAM numbers for each.
import numpy as np, rasterio, tempfile, json, shutil
from rasterio.transform import from_origin
from pathlib import Path
from skyvonyx.config import JobConfig
from skyvonyx.pipeline import run_job

tmp = Path(tempfile.mkdtemp()); (tmp / "input").mkdir()
rng = np.random.default_rng(0)
arr = (rng.integers(40, 90, (3, 1500, 2200), dtype=np.int16)).astype(np.uint8)
arr[:, 700:730, 900:1000] = 230       # a bright "ship"-shaped blob across a tile boundary
arr[:, 200:240, 300:340] = 230
with rasterio.open(tmp / "input" / "smoke.tif", "w", driver="GTiff", height=1500, width=2200,
                   count=3, dtype="uint8", crs="EPSG:32643",
                   transform=from_origin(300000, 2100000, 10, 10)) as dst:   # UTM 43N, 10 m px
    dst.write(arr)

cfg = JobConfig(job_id="smoke", detector="yolov8", segmenter="sam2",
                detector_weights=DETECTOR_WEIGHTS, segmenter_weights=str(SAM2_CKPT),
                tile=1024, overlap=0.2, conf=0.05, batch=8)
(tmp / "job.json").write_text(json.dumps(cfg.to_dict()))
summary = run_job(tmp, cfg)

img = summary["images"][0]
print("\nscene:", img["width"], "x", img["height"], "| tiles", img["tiles_total"], "| georeferenced", img["georeferenced"], img["crs"])
print("detections:", img["detections_before_merge"], "->", img["detections_after_merge"], "after", img["merge_backend"])
print("polygons from SAM 2:", img["polygons_from_masks"])
print("stage seconds:", summary["stage_seconds"])
print("\nscheduler:")
for ev in summary["scheduler"]["events"]:
    print(f"  {ev['expert']:<10} load {ev['load_s']}s  evicted={ev['evicted']}  VRAM peak {ev['vram_peak_mb']} MB")
print("peak VRAM for the whole run:", summary["scheduler"]["peak_vram_mb"], "MB of", summary["scheduler"]["vram_now"]["total_mb"])
print("outputs:", list(img["outputs"].values()))
# COCO yolov8n will find little or nothing on synthetic blobs - that is expected. The point of this
# cell is that the GPU path runs, VRAM is released between experts, and a WGS84 GeoJSON is written.

In [ ]:
#@title 7. (Optional) Submit a job from this notebook
# Normally jobs arrive from the website/API or by copying a folder into inbox/. This helper does the
# same thing by hand: copies images, writes job.json, creates READY last.
from skyvonyx.queue_worker import create_job
# create_job(ROOT, "demo_001", ["/content/drive/MyDrive/scenes/scene.tif"],
#            customer="internal", detector="yolov8", segmenter="sam2",
#            detector_weights=DETECTOR_WEIGHTS, segmenter_weights=str(SAM2_CKPT),
#            sensor="optical", conf=0.25, tile=1024, overlap=0.2)
print(queue_state(ROOT))

In [ ]:
#@title 8. Run the worker (blocks until STOP or max_hours; safe to interrupt)
import os
from skyvonyx.queue_worker import run_forever
# Every job's job.json may set its own weights; these are the defaults a job falls back to
# when it does not. Edit a job.json instead of this cell to change one job's settings.
os.environ.setdefault("SKYVONYX_DEFAULT_DETECTOR_WEIGHTS", DETECTOR_WEIGHTS)
os.environ.setdefault("SKYVONYX_DEFAULT_SAM2_CKPT", str(SAM2_CKPT))

run_forever(ROOT, poll_s=30, max_hours=11.0, requeue_on_start=True)
# Stop cleanly from another tab: create the file  MyDrive/skyvonyx/worker/STOP

## Reviewing a job

1. Open `jobs/review/<job_id>/output/REPORT_DRAFT.md` and `*_preview.png`.
2. Load `*_wgs84.geojson` over the source scene in QGIS. Shapes on the objects? Count a few.
3. Fix or delete false positives, add misses, set `status` to `verified` on what you checked.
4. Create an empty file named **`APPROVED`** in the job folder (or `REJECTED` containing a reason).
5. The worker moves it to `done/` on its next tick. **Delivery is manual**: share `output/` with the customer yourself.

The report says *accuracy NOT MEASURED* until a reviewer records precision, recall, IoU and second-reviewer agreement on a gold subset. Keep it that way.

## Resuming after a disconnect

Just re-run cells 1-4 and 8. `requeue_on_start=True` pushes anything stuck in `processing/` back to `inbox/`, and
the stages that already finished are read back from `<job>/memory/` - a job that died during SAM 2 restarts at SAM 2.